# Исследование поискового запроса

Одна Hydra-композиция: посмотреть разбор и DSL, выполнить одиночный поиск,
затем прогнать CSV `id,text` через тот же pipeline и явно сохранить выдачи.

Откройте notebook в Jupyter/VS Code с Python kernel окружения проекта (Python 3.12+).
Jupyter предоставляется вашим окружением; runtime проекта его не устанавливает.
Notebook можно открыть из корня checkout или вложенного каталога.

В ячейке параметров замените пример URL, индекс и поля на значения вашего контура.
Authorization берётся из `OPENSEARCH_AUTHORIZATION` в окружении kernel.
Не вставляйте credentials в notebook или overrides. Не выводите resolved config.
Результаты ячеек содержат реальные документы; поставляемый notebook не содержит outputs.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs/main.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.core.transformers.data import PipelineData
from src.domain.request.filters import QueryFilter

## Параметры опыта

Это пример mapping, а не подтверждённая схема вашего индекса. Для одного text-поля
укажите его в `fields`, а `title_field` оставьте `null`. В `source_overrides` можно
добавить overrides словарей, parsing/retrieval/graph groups, весов и глубины.
Для fuzzy нужны graph `fuzzy` и pipeline `fuzzy`.

`pipeline` выбирает полный e2e путь. Все пути ниже относятся к checkout;
для собственного набора можно задать абсолютный путь. Выходной файл не перезаписывается.


In [ ]:
source_overrides = [
    "connection.url=http://localhost:9200",
    "jobboard.source.index=vacancies",
    "jobboard.source.fields=[title,description]",
    "jobboard.source.title_field=title",
]
pipeline = "parsed"  # baseline, lexical, parsed, fused; fuzzy с graph/fuzzy
text = "python backend developer"
input_path = ROOT / "examples/queries.csv"
output_path = ROOT / "outputs" / "notebook" / "searches.csv"

## Сборка эксперимента

Compose используется как context manager: ячейку можно выполнять повторно.
Exporters отключены, поэтому нет зависимости от CLI runtime Hydra и автоматической
записи файлов. После изменения параметров повторите эту ячейку: старые объекты
не меняются при редактировании config.


In [ ]:
with initialize_config_dir(config_dir=str(ROOT / "configs"), version_base="1.3"):
    cfg = compose(
        config_name="main",
        overrides=[
            "~experiments/vacancy-search/interactive@experiment",
            "+experiments/vacancy-search/jobboard/lexical@experiment=001-interactive",
            *source_overrides,
            f"experiment.pipeline={pipeline}",
            "experiment.exporters=[]",
        ],
    )
experiment = instantiate(cfg.experiment, _convert_="all")

## Разбор и DSL без HTTP

Исследуется конкретный поисковый лист `parsed_leaf`. Для lexical выбирайте
`lexical_leaf`, для baseline — `baseline_leaf` (у него нет parser).
Fused выполняет lexical и parsed; его два запроса смотрятся отдельно.
`build_query` использует тот же compiler, что и настоящий поиск.

Payload для просмотра включает context и статические filters текущего конфига.
Дополнительные условия ручного поиска нужно включать и в этот payload.


In [ ]:
leaf = experiment.registry["parsed_leaf"]
parsed = leaf.parser.parse(text)
probe = PipelineData(
    context=instantiate(cfg.experiment.context, _convert_="all"),
    filters=[*instantiate(cfg.experiment.filters, _convert_="all"), QueryFilter(text)],
)
dsl = leaf.build_query(probe)
print(parsed)
print(json.dumps(dsl, ensure_ascii=False, indent=2))

## Один e2e поиск

Эта ячейка обращается к вашему OpenSearch. Метод применяет context/filters
эксперимента и возвращает независимую SERP. Его можно вызывать повторно,
в том числе после неудачного запроса. Дополнительные условия задаются через
`search(text, context={...}, filters=[...])`: context переопределяет свойства
верхнего уровня, filters добавляются к статическим условиям.


In [ ]:
serp = experiment.search(text)
rows = [
    {"id": item.id, "score": item.score, "fields": item.fields} for item in serp.items
]
rows

## Батч без разметки релевантности

Входной CSV имеет ровно колонки `id,text`; id уникальны, оба значения непустые.
Сначала проверяется весь CSV, затем запросы выполняются последовательно.
`run()` возвращает снимок результатов без записи файла.

В этой ячейке используется тот же собранный `experiment`, поэтому сохраняются
выбранный pipeline, context, filters и параметры. Настройки батча также задаются
через Hydra. Для CLI готов отдельный конфиг `003-batch`; команда — в README.


In [ ]:
batch_cfg = OmegaConf.create(
    {
        "_target_": "src.experiments.batch.search.BatchSearchExperiment",
        "input_path": str(input_path),
        "exporters": [
            {
                "_target_": "src.components.reporting.exporters.csv.SearchCSVExporter",
                "path": str(output_path),
            }
        ],
    }
)
batch = instantiate(batch_cfg, search=experiment, _convert_="all")
artifact = batch.run()
artifact.to_rows()[:5]

## Явное сохранение CSV

Одна строка на документ: `query_id,query,rank,document_id,score,fields`.
`fields` — полный JSON с вложенными полями. Пустая выдача сохраняется одной
строкой с пустыми полями результата; null score — пустая ячейка.
Существующий файл не перезаписывается. При повторном опыте выберите новый путь
и пересоберите batch. Ошибка любого поиска не создаёт частичный CSV.


In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
for exporter in batch.exporters:
    exporter.export(artifact)
output_path